# MemAE on MVTec AD — Kaggle notebook

This notebook trains **one Memory-Augmented Autoencoder (MemAE) per MVTec AD category**. Each model is trained only on defect-free `train/good` images. A held-out subset of those normal images is used for early stopping and to calibrate image- and pixel-level F1 thresholds. The official test folders are used only for final evaluation.

**Qualitative figures:** one normal example and one example from every defect subtype, each with input, ground-truth mask, anomaly map and predicted-mask overlay. **Outputs:** category checkpoints, training histories, per-image scores, compressed anomaly maps and masks, qualitative visualisations, category-wise metrics, macro summary, and a progress/failure log. Results are stored separately under `CS663Project/memae_runs` on Google Drive when the Kaggle secret `GDRIVE_TOKEN` is available.

MemAE follows the memory-augmented reconstruction idea of Gong et al. (ICCV 2019): an encoder creates spatial feature queries, a learnable memory stores normal feature patterns, sparse memory addressing retrieves features, and a decoder reconstructs the input. At inference, reconstruction error provides the anomaly map. [Paper](https://openaccess.thecvf.com/content_ICCV_2019/papers/Gong_Memorizing_Normality_to_Detect_Anomaly_Memory-Augmented_Deep_Autoencoder_for_Unsupervised_ICCV_2019_paper.pdf).

**Evaluation note:** all images and masks are resized to 256×256 for this experiment. Image score is the mean of the largest 1% of reconstruction-error pixels. Image F1 and pixel F1 use thresholds estimated from held-out normal training images, not from test labels. AUROC and AP are threshold-independent.

## 1. Environment and experiment configuration

In [ ]:
import os
import sys
import json
import time
import math
import random
import shutil
import tarfile
import subprocess
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MEMORY_SIZE = 256
SHRINK_THRESHOLD = 1e-3
TEMPERATURE = 0.07
ENTROPY_WEIGHT = 1e-4
VALIDATION_FRACTION = 0.10
IMAGE_THRESHOLD_PERCENTILE = 99.0
PIXEL_THRESHOLD_PERCENTILE = 99.5
NUM_WORKERS = 2
SAVE_RAW_MAPS = True

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

LOCAL_ROOT = Path("/kaggle/working/memae_runs")
CATEGORIES_ROOT = LOCAL_ROOT / "categories"
METRICS_ROOT = LOCAL_ROOT / "metrics"
VIS_ROOT = LOCAL_ROOT / "visualizations"
for folder in [CATEGORIES_ROOT, METRICS_ROOT, VIS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print("Local output:", LOCAL_ROOT)

## 2. Find or extract MVTec AD

In [ ]:
def is_mvtec_root(path):
    path = Path(path)
    return all((path / category / "train" / "good").is_dir() for category in CATEGORIES)

candidate_roots = [
    Path("/kaggle/working/mvtec_extracted"),
    Path("/kaggle/working/mvtec_extracted/mvtec_anomaly_detection"),
    Path("/kaggle/input/datasets/anrdisded/mvtecac/mvtec_anomaly_detection"),
]

DATA_ROOT = next((p for p in candidate_roots if is_mvtec_root(p)), None)

if DATA_ROOT is None:
    archive_candidates = [
        Path("/kaggle/input/datasets/anrdisded/mvtecac/mvtec_anomaly_detection.tar.xz")
    ]
    archive_candidates += [
        Path(p) for p in glob.glob("/kaggle/input/**/*.tar.xz", recursive=True)
        if "mvtec" in Path(p).name.lower()
    ]
    archive = next((p for p in archive_candidates if p.is_file()), None)
    if archive is None:
        raise FileNotFoundError(
            "Could not find the MVTec AD folder or .tar.xz archive under /kaggle/input. "
            "Check that the MVTec Kaggle dataset is attached to this notebook."
        )

    extract_root = Path("/kaggle/working/mvtec_extracted")
    extract_root.mkdir(parents=True, exist_ok=True)
    print("Extracting:", archive)
    with tarfile.open(archive, "r:*") as tar:
        try:
            tar.extractall(path=extract_root, filter="data")
        except TypeError:
            tar.extractall(path=extract_root)

    possible = [extract_root, extract_root / "mvtec_anomaly_detection"]
    DATA_ROOT = next((p for p in possible if is_mvtec_root(p)), None)

if DATA_ROOT is None:
    for root, dirs, files in os.walk("/kaggle/working/mvtec_extracted"):
        root_path = Path(root)
        if is_mvtec_root(root_path):
            DATA_ROOT = root_path
            break

if DATA_ROOT is None:
    raise FileNotFoundError("MVTec extraction finished, but the 15-category dataset root was not located.")

print("Dataset root:", DATA_ROOT)
for category in CATEGORIES:
    count = len(list((DATA_ROOT / category / "train" / "good").glob("*.png")))
    print(f"{category:12s} {count:4d} normal training images")

## 3. Optional Google Drive backup and resume

The notebook uses the Kaggle secret `GDRIVE_TOKEN`, which you already configured for the PatchCore backup. If the secret or network connection is unavailable, training still runs locally; enable Drive backup later and rerun the backup cell.

In [ ]:
DRIVE_ENABLED = False
DRIVE_REMOTE_ROOT = "gdrive:CS663Project/memae_runs"

try:
    from kaggle_secrets import UserSecretsClient
    drive_token = UserSecretsClient().get_secret("GDRIVE_TOKEN")

    if not shutil.which("rclone"):
        subprocess.run(
            "curl -fsSL https://rclone.org/install.sh | bash",
            shell=True,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )

    rclone_config_dir = Path("/root/.config/rclone")
    rclone_config_dir.mkdir(parents=True, exist_ok=True)
    rclone_config_path = rclone_config_dir / "rclone.conf"
    rclone_config_path.write_text(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"token = {drive_token}\n"
    )
    os.chmod(rclone_config_path, 0o600)

    subprocess.run(["rclone", "mkdir", DRIVE_REMOTE_ROOT], check=True, capture_output=True, text=True)
    subprocess.run(["rclone", "mkdir", f"{DRIVE_REMOTE_ROOT}/categories"], check=True, capture_output=True, text=True)
    subprocess.run(["rclone", "mkdir", f"{DRIVE_REMOTE_ROOT}/metrics"], check=True, capture_output=True, text=True)

    DRIVE_ENABLED = True
    print("Google Drive backup enabled:", DRIVE_REMOTE_ROOT)

    subprocess.run(
        ["rclone", "copy", f"{DRIVE_REMOTE_ROOT}/categories", str(CATEGORIES_ROOT), "--transfers", "4", "--checkers", "8"],
        check=True
    )
    subprocess.run(
        ["rclone", "copy", f"{DRIVE_REMOTE_ROOT}/metrics", str(METRICS_ROOT), "--transfers", "4", "--checkers", "8"],
        check=True
    )
    print("Existing Drive results restored to this Kaggle session.")
except Exception as error:
    DRIVE_ENABLED = False
    print("Google Drive backup unavailable; continuing with local outputs.")
    print(type(error).__name__ + ":", str(error)[:500])

In [ ]:
def backup_path(local_path, remote_path):
    if not DRIVE_ENABLED:
        return False

    print(f"Starting backup: {local_path}", flush=True)
    try:
        subprocess.run(
            [
                "rclone", "copy", str(local_path), remote_path,
                "--transfers", "2", "--checkers", "4",
                "--stats", "30s", "--stats-one-line"
            ],
            check=True,
            timeout=900
        )
        print("Backup completed.", flush=True)
        return True
    except subprocess.TimeoutExpired:
        print("Backup warning: timed out after 15 minutes.", flush=True)
        return False
    except Exception as error:
        print("Backup warning:", type(error).__name__, str(error)[:300], flush=True)
        return False


def update_progress_files():
    rows = []
    failures = []

    for category in CATEGORIES:
        metrics_file = CATEGORIES_ROOT / category / "metrics.json"
        if metrics_file.exists():
            with open(metrics_file) as file:
                result = json.load(file)
            if result.get("completed"):
                rows.append(result)

    failure_file = METRICS_ROOT / "failures.json"
    if failure_file.exists():
        with open(failure_file) as file:
            failures = json.load(file)

    if rows:
        table = pd.DataFrame(rows).sort_values("category")
        table.to_csv(METRICS_ROOT / "all_categories_metrics.csv", index=False)
        metric_columns = [
            "image_AUROC", "image_AP", "image_F1Score",
            "pixel_AUROC", "pixel_AP", "pixel_F1Score"
        ]
        summary = table[metric_columns].agg(["mean", "std", "median"]).T
        summary.to_csv(METRICS_ROOT / "aggregate_summary.csv")

    completed = sorted([row["category"] for row in rows])
    progress = {
        "completed_categories": completed,
        "completed_count": len(completed),
        "total_categories": len(CATEGORIES),
        "remaining_categories": [c for c in CATEGORIES if c not in completed],
        "failed_categories": failures,
    }
    with open(METRICS_ROOT / "progress.json", "w") as file:
        json.dump(progress, file, indent=2)

    return progress


## 4. Dataset classes and preprocessing

In [ ]:
def load_rgb(path):
    image = Image.open(path).convert("RGB")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


def load_mask(path):
    image = Image.open(path).convert("L")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
    array = (np.asarray(image, dtype=np.uint8) > 0).astype(np.float32)
    return torch.from_numpy(array).unsqueeze(0)


class NormalImageDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = list(image_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        return load_rgb(self.image_paths[index])


class MVTecTestDataset(Dataset):
    def __init__(self, root, category):
        self.root = Path(root)
        self.category = category
        self.base = self.root / category
        self.entries = []

        for image_path in sorted((self.base / "test").glob("*/*.png")):
            defect_type = image_path.parent.name
            label = int(defect_type != "good")
            if label:
                mask_path = self.base / "ground_truth" / defect_type / f"{image_path.stem}_mask.png"
                if not mask_path.is_file():
                    raise FileNotFoundError(f"Missing ground-truth mask: {mask_path}")
            else:
                mask_path = None
            self.entries.append((image_path, mask_path, label, defect_type))

        if not self.entries:
            raise RuntimeError(f"No test images found for {category}")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        image_path, mask_path, label, defect_type = self.entries[index]
        image = load_rgb(image_path)
        mask = load_mask(mask_path) if mask_path is not None else torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
        return image, mask, label, str(image_path), defect_type


def split_normal_paths(category):
    paths = sorted((DATA_ROOT / category / "train" / "good").glob("*.png"))
    if len(paths) < 5:
        raise RuntimeError(f"Too few normal training images in {category}: {len(paths)}")

    rng = np.random.default_rng(SEED + CATEGORIES.index(category))
    indices = rng.permutation(len(paths))
    val_count = max(2, int(round(len(paths) * VALIDATION_FRACTION)))
    val_count = min(val_count, len(paths) - 2)
    val_indices = set(indices[:val_count].tolist())
    train_paths = [path for i, path in enumerate(paths) if i not in val_indices]
    val_paths = [path for i, path in enumerate(paths) if i in val_indices]
    return train_paths, val_paths

## 5. MemAE architecture

The encoder downsamples the 256×256 image to a 32×32 spatial feature map with 128 channels. Each spatial feature is treated as a query against a learnable bank of memory slots. Softmax addressing is sparsified with hard shrinkage and renormalized; a max-attention fallback prevents all-zero memory reads. The decoder reconstructs from the retrieved memory features.

In [ ]:
class MemoryModule(nn.Module):
    def __init__(self, channels=128, memory_size=256, shrink_threshold=1e-3, temperature=0.07):
        super().__init__()
        self.memory = nn.Parameter(torch.empty(memory_size, channels))
        nn.init.xavier_uniform_(self.memory)
        self.shrink_threshold = shrink_threshold
        self.temperature = temperature

    def forward(self, feature_map):
        batch, channels, height, width = feature_map.shape
        queries = feature_map.permute(0, 2, 3, 1).reshape(-1, channels)
        normalized_queries = F.normalize(queries, dim=1)
        normalized_memory = F.normalize(self.memory, dim=1)

        similarities = normalized_queries @ normalized_memory.t()
        soft_attention = torch.softmax(similarities / self.temperature, dim=1)
        sparse_attention = F.relu(soft_attention - self.shrink_threshold)
        denominator = sparse_attention.sum(dim=1, keepdim=True)
        fallback = F.one_hot(
            soft_attention.argmax(dim=1), num_classes=soft_attention.shape[1]
        ).to(dtype=soft_attention.dtype)
        sparse_attention = torch.where(
            denominator > 1e-12,
            sparse_attention / denominator.clamp_min(1e-12),
            fallback
        )

        retrieved = sparse_attention @ self.memory
        entropy = -(soft_attention * torch.log(soft_attention.clamp_min(1e-12))).sum(dim=1).mean()
        retrieved = retrieved.reshape(batch, height, width, channels).permute(0, 3, 1, 2)
        return retrieved, entropy


class MemoryAutoencoder(nn.Module):
    def __init__(self, memory_size=MEMORY_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(128, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.1, inplace=True)
        )
        self.memory = MemoryModule(
            channels=128,
            memory_size=memory_size,
            shrink_threshold=SHRINK_THRESHOLD,
            temperature=TEMPERATURE
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Sigmoid()
        )

    def forward(self, images):
        encoded = self.encoder(images)
        memory_features, entropy = self.memory(encoded)
        reconstruction = self.decoder(memory_features)
        return reconstruction, entropy


def memae_loss(reconstruction, target, entropy):
    reconstruction_loss = F.mse_loss(reconstruction, target)
    total_loss = reconstruction_loss + ENTROPY_WEIGHT * entropy
    return total_loss, reconstruction_loss

## 6. Training, validation and checkpointing

In [ ]:
def train_category(category):
    category_output = CATEGORIES_ROOT / category
    category_output.mkdir(parents=True, exist_ok=True)

    train_paths, val_paths = split_normal_paths(category)
    train_loader = DataLoader(
        NormalImageDataset(train_paths), batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0
    )
    val_loader = DataLoader(
        NormalImageDataset(val_paths), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0
    )

    category_seed = SEED + CATEGORIES.index(category)
    random.seed(category_seed)
    np.random.seed(category_seed)
    torch.manual_seed(category_seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(category_seed)

    model = MemoryAutoencoder(MEMORY_SIZE).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_epoch = 0
    patience_count = 0
    history = []
    checkpoint_path = category_output / "best_model.pt"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for images in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            reconstruction, entropy = model(images)
            loss, _ = memae_loss(reconstruction, images, entropy)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for images in val_loader:
                images = images.to(DEVICE, non_blocking=True)
                reconstruction, entropy = model(images)
                loss, _ = memae_loss(reconstruction, images, entropy)
                val_losses.append(float(loss.cpu()))

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "validation_loss": val_loss})
        print(
            f"{category} | epoch {epoch:02d}/{EPOCHS} | "
            f"train={train_loss:.6f} | val={val_loss:.6f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_count = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "category": category,
                "image_size": IMAGE_SIZE,
                "memory_size": MEMORY_SIZE,
                "best_epoch": best_epoch,
                "best_validation_loss": best_val_loss,
                "seed": category_seed,
            }, checkpoint_path)
        else:
            patience_count += 1

        pd.DataFrame(history).to_csv(category_output / "training_history.csv", index=False)
        if patience_count >= PATIENCE:
            print("Early stopping triggered.")
            break

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint, val_paths

## 7. Calibrate thresholds using held-out normal images

The held-out validation subset contains only `train/good` images. The image threshold is the 99th percentile of normal image scores; the pixel threshold is the 99.5th percentile of normal reconstruction-error pixels. No anomalous test labels are used to select thresholds.

In [ ]:
def image_score_from_map(anomaly_map):
    flattened = anomaly_map.reshape(anomaly_map.shape[0], -1)
    k = max(1, int(math.ceil(0.01 * flattened.shape[1])))
    return torch.topk(flattened, k=k, dim=1).values.mean(dim=1)


def calibrate_thresholds(model, val_paths):
    loader = DataLoader(
        NormalImageDataset(val_paths), batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0
    )
    image_scores = []
    pixel_values = []
    model.eval()
    with torch.no_grad():
        for images in loader:
            images = images.to(DEVICE, non_blocking=True)
            reconstruction, _ = model(images)
            anomaly_maps = (images - reconstruction).pow(2).mean(dim=1)
            scores = image_score_from_map(anomaly_maps)
            image_scores.extend(scores.cpu().numpy().tolist())
            pixel_values.append(anomaly_maps.cpu().numpy().reshape(-1))

    image_threshold = float(np.percentile(image_scores, IMAGE_THRESHOLD_PERCENTILE))
    all_pixel_values = np.concatenate(pixel_values)
    pixel_threshold = float(np.percentile(all_pixel_values, PIXEL_THRESHOLD_PERCENTILE))
    return {
        "image_threshold": image_threshold,
        "pixel_threshold": pixel_threshold,
        "image_threshold_percentile": IMAGE_THRESHOLD_PERCENTILE,
        "pixel_threshold_percentile": PIXEL_THRESHOLD_PERCENTILE,
        "calibration_images": len(val_paths),
    }

## 8. Evaluation and artifact saving

In [ ]:
def safe_auc(labels, scores):
    labels = np.asarray(labels).astype(np.uint8)
    if np.unique(labels).size < 2:
        return float("nan")
    return float(roc_auc_score(labels, scores))


def safe_ap(labels, scores):
    labels = np.asarray(labels).astype(np.uint8)
    if labels.sum() == 0:
        return float("nan")
    return float(average_precision_score(labels, scores))


def evaluate_category(category, model, checkpoint, val_paths, runtime_minutes):
    category_output = CATEGORIES_ROOT / category
    calibration = calibrate_thresholds(model, val_paths)
    dataset = MVTecTestDataset(DATA_ROOT, category)
    loader = DataLoader(
        dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0
    )

    image_labels = []
    image_scores = []
    pixel_labels = []
    pixel_scores = []
    anomaly_maps = []
    ground_truth_masks = []
    image_paths = []
    records = []

    model.eval()
    with torch.no_grad():
        for images, masks, labels, paths, defect_types in tqdm(loader, desc=f"Evaluating {category}"):
            images = images.to(DEVICE, non_blocking=True)
            reconstruction, _ = model(images)
            batch_maps = (images - reconstruction).pow(2).mean(dim=1)
            batch_scores = image_score_from_map(batch_maps)

            maps_np = batch_maps.cpu().numpy()
            masks_np = masks[:, 0].numpy()
            labels_np = labels.numpy().astype(np.uint8)
            scores_np = batch_scores.cpu().numpy()

            image_labels.extend(labels_np.tolist())
            image_scores.extend(scores_np.tolist())
            pixel_labels.append(masks_np.reshape(-1).astype(np.uint8))
            pixel_scores.append(maps_np.reshape(-1).astype(np.float32))
            anomaly_maps.extend([item.astype(np.float16) for item in maps_np])
            ground_truth_masks.extend([item.astype(np.uint8) for item in masks_np])
            image_paths.extend(list(paths))

            for i in range(len(paths)):
                records.append({
                    "image_path": paths[i],
                    "defect_type": defect_types[i],
                    "label": int(labels_np[i]),
                    "image_score_top1pct": float(scores_np[i]),
                    "image_prediction": int(scores_np[i] >= calibration["image_threshold"]),
                })

    all_pixel_labels = np.concatenate(pixel_labels)
    all_pixel_scores = np.concatenate(pixel_scores)
    pixel_predictions = (all_pixel_scores >= calibration["pixel_threshold"]).astype(np.uint8)

    metrics = {
        "category": category,
        "completed": True,
        "status": "completed",
        "runtime_minutes": round(float(runtime_minutes), 2),
        "test_images": int(len(image_labels)),
        "normal_images": int(np.sum(np.asarray(image_labels) == 0)),
        "defective_images": int(np.sum(np.asarray(image_labels) == 1)),
        "best_epoch": int(checkpoint["best_epoch"]),
        "best_validation_loss": float(checkpoint["best_validation_loss"]),
        "image_AUROC": safe_auc(image_labels, image_scores),
        "image_AP": safe_ap(image_labels, image_scores),
        "image_F1Score": float(f1_score(image_labels, np.asarray(image_scores) >= calibration["image_threshold"], zero_division=0)),
        "pixel_AUROC": safe_auc(all_pixel_labels, all_pixel_scores),
        "pixel_AP": safe_ap(all_pixel_labels, all_pixel_scores),
        "pixel_F1Score": float(f1_score(all_pixel_labels, pixel_predictions, zero_division=0)),
        "image_threshold": calibration["image_threshold"],
        "pixel_threshold": calibration["pixel_threshold"],
        "image_threshold_percentile": calibration["image_threshold_percentile"],
        "pixel_threshold_percentile": calibration["pixel_threshold_percentile"],
        "calibration_images": calibration["calibration_images"],
        "image_score_definition": "mean of largest 1% of reconstruction-error pixels",
        "image_size": IMAGE_SIZE,
    }

    pd.DataFrame(records).to_csv(category_output / "per_image_predictions.csv", index=False)
    if SAVE_RAW_MAPS:
        np.savez_compressed(
            category_output / "anomaly_maps.npz",
            anomaly_maps=np.stack(anomaly_maps),
            masks=np.stack(ground_truth_masks),
            image_labels=np.asarray(image_labels, dtype=np.uint8),
            image_scores=np.asarray(image_scores, dtype=np.float32),
            image_paths=np.asarray(image_paths),
        )

    with open(category_output / "calibration.json", "w") as file:
        json.dump(calibration, file, indent=2)
    with open(category_output / "metrics.json", "w") as file:
        json.dump(metrics, file, indent=2)

    return metrics

## 9. Qualitative visualisations

In [ ]:
def save_category_visualization(category):
    category_output = CATEGORIES_ROOT / category
    raw_path = category_output / "anomaly_maps.npz"
    predictions_path = category_output / "per_image_predictions.csv"

    if not raw_path.exists() or not predictions_path.exists():
        print("Visualisation skipped; prediction artifacts missing for", category)
        return

    data = np.load(raw_path, allow_pickle=True)
    predictions = pd.read_csv(predictions_path)

    labels = predictions["label"].to_numpy().astype(int)
    scores = predictions["image_score_top1pct"].to_numpy()
    defect_types = predictions["defect_type"].fillna("unknown").astype(str).to_numpy()

    chosen = []
    normal_indices = np.where(labels == 0)[0]
    if len(normal_indices):
        normal_index = int(normal_indices[np.argmax(scores[normal_indices])])
        chosen.append((normal_index, "Normal"))

    for defect_type in sorted(set(defect_types[labels == 1])):
        subtype_indices = np.where((labels == 1) & (defect_types == defect_type))[0]
        if len(subtype_indices):
            best_index = int(subtype_indices[np.argmax(scores[subtype_indices])])
            chosen.append((best_index, defect_type))

    if not chosen:
        print("No samples available for visualisation:", category)
        return

    fig, axes = plt.subplots(
        len(chosen), 4,
        figsize=(14, 3.2 * len(chosen)),
        squeeze=False
    )

    with open(category_output / "calibration.json") as file:
        threshold = float(json.load(file)["pixel_threshold"])

    for row, (index, sample_label) in enumerate(chosen):
        image_path = data["image_paths"][index]
        image = Image.open(image_path).convert("RGB")
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        image_array = np.asarray(image)

        gt = data["masks"][index]
        amap = data["anomaly_maps"][index].astype(np.float32)
        pred = (amap >= threshold).astype(np.uint8)

        axes[row, 0].imshow(image_array)
        axes[row, 0].set_title("Input")

        axes[row, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground truth mask")

        axes[row, 2].imshow(image_array)
        axes[row, 2].imshow(amap, cmap="inferno", alpha=0.55)
        axes[row, 2].set_title("MemAE anomaly map")

        overlay = image_array.astype(np.float32).copy()
        predicted_pixels = pred.astype(bool)
        magenta = np.array([255.0, 0.0, 255.0], dtype=np.float32)
        overlay[predicted_pixels] = (
            0.25 * overlay[predicted_pixels] + 0.75 * magenta
        )
        axes[row, 3].imshow(np.clip(overlay, 0, 255).astype(np.uint8))
        axes[row, 3].set_title("Predicted mask")

        for column in range(4):
            axes[row, column].axis("off")

        axes[row, 0].set_ylabel(sample_label)

    fig.suptitle(
        f"MemAE qualitative results — {category}",
        fontsize=14
    )
    fig.tight_layout()

    destination = VIS_ROOT / f"{category}_examples.png"
    fig.savefig(destination, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", destination)


## 10. Run all categories (resumable)

Completed categories are detected from their saved `metrics.json`. When Drive backup is enabled, completed category folders are restored at startup and each finished category is uploaded immediately. If a category fails, it is logged and the loop continues.

In [ ]:
failures_path = METRICS_ROOT / "failures.json"
if failures_path.exists():
    with open(failures_path) as file:
        failures = json.load(file)
else:
    failures = []

for category_index, category in enumerate(CATEGORIES):
    category_output = CATEGORIES_ROOT / category
    metrics_file = category_output / "metrics.json"

    if metrics_file.exists():
        try:
            with open(metrics_file) as file:
                existing = json.load(file)
        except Exception:
            existing = {}
        if existing.get("completed") is True:
            print(f"Skipping {category}: completed results already exist.", flush=True)
            continue

    started = time.time()
    print("\n" + "=" * 72, flush=True)
    print(f"CATEGORY {category_index + 1}/{len(CATEGORIES)}: {category}", flush=True)
    print("=" * 72, flush=True)

    try:
        model, checkpoint, val_paths = train_category(category)
        runtime_minutes = (time.time() - started) / 60.0
        metrics = evaluate_category(category, model, checkpoint, val_paths, runtime_minutes)

        with open(metrics_file, "w") as file:
            json.dump(metrics, file, indent=2)

        save_category_visualization(category)
        print(json.dumps(metrics, indent=2), flush=True)

        print("Updating progress files...", flush=True)
        update_progress_files()

        print("Backing up category artifacts...", flush=True)
        backup_path(category_output, f"{DRIVE_REMOTE_ROOT}/categories/{category}")

        print("Backing up metrics...", flush=True)
        backup_path(METRICS_ROOT, f"{DRIVE_REMOTE_ROOT}/metrics")

        visual_file = VIS_ROOT / f"{category}_examples.png"
        if visual_file.exists():
            print("Backing up this category's visualisation...", flush=True)
            backup_path(visual_file, f"{DRIVE_REMOTE_ROOT}/visualizations")

        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    except Exception as error:
        failures = [item for item in failures if item.get("category") != category]
        failures.append({"category": category, "error": repr(error)})
        with open(failures_path, "w") as file:
            json.dump(failures, file, indent=2)
        print("CATEGORY FAILED:", category, repr(error), flush=True)
        update_progress_files()
        backup_path(METRICS_ROOT, f"{DRIVE_REMOTE_ROOT}/metrics")

progress = update_progress_files()
backup_path(METRICS_ROOT, f"{DRIVE_REMOTE_ROOT}/metrics")
print(json.dumps(progress, indent=2), flush=True)


## 11. Final result tables and aggregate summary

In [ ]:
progress = update_progress_files()
metrics_file = METRICS_ROOT / "all_categories_metrics.csv"
if metrics_file.exists():
    results = pd.read_csv(metrics_file)
    display(results.sort_values("category").reset_index(drop=True))

    metric_columns = [
        "image_AUROC", "image_AP", "image_F1Score",
        "pixel_AUROC", "pixel_AP", "pixel_F1Score"
    ]
    print("Macro average across completed categories:")
    display(results[metric_columns].mean().to_frame("macro_mean").T)
    print("Completed:", progress["completed_count"], "/", progress["total_categories"])
else:
    print("No completed categories yet.")

print("Local results:", LOCAL_ROOT)
print("Google Drive enabled:", DRIVE_ENABLED)

## 12. References and experiment notes

1. Dong Gong, Lingqiao Liu, Vuong Le, Budhaditya Saha, Moussa Reda Mansour, Svetha Venkatesh, and Anton van den Hengel. **“Memorizing Normality to Detect Anomaly: Memory-Augmented Deep Autoencoder for Unsupervised Anomaly Detection.”** ICCV, 2019. [Open-access paper](https://openaccess.thecvf.com/content_ICCV_2019/html/Gong_Memorizing_Normality_to_Detect_Anomaly_Memory-Augmented_Deep_Autoencoder_for_Unsupervised_ICCV_2019_paper.html).
2. **MVTec AD** dataset: [official dataset page](https://www.mvtec.com/research-teaching/datasets/mvtec-ad).

### Important interpretation notes

- This is a reproducible MemAE implementation based on the paper's memory-addressing principle, not a claim of an exact reproduction of every setting in the original paper.
- F1 thresholds are calibrated on held-out normal training images only. F1 values are therefore tied to the stated calibration percentiles.
- AUROC and AP are calculated over each category's complete test set. Aggregate values are macro averages across categories, matching the category-wise reporting style used for PatchCore.
- Pixel metrics are computed at 256×256 after resizing both prediction maps and ground-truth masks.
- Keep MemAE outputs separate from PatchCore outputs. Do not overwrite `CS663Project/PatchCore_Results`.